# Formula 1 Podium Prediction — Milestone 1
**CSEN 903 Advanced Computer Lab — Team [40]**

**Goal:** predict, for each driver in each race, whether they finish on the podium (top 3),
using only information available after qualifying and before the race starts.

1. Setup & Data Loading
2. Table-by-Table Audit & Cleaning (EDA → cleaning → after-check for each table)
3. Cross-Table Checks & Cleaning (foreign keys, grain, era decisions, joins)
4. EDA on the Combined Data
5. Data-Engineering Questions
6. Feature Engineering & Selection
7. Train / Validation / Test Split
8. Modeling
9. Experiments
10. Explainability (XAI)
11. Inference Function

# 1. Setup & Data Loading
## 1.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

np.random.seed(42)

## 1.2 Load the 14 raw tables
Each CSV is loaded into its own DataFrame. These raw tables are never modified; all cleaning is done on copies.

In [ ]:
path = '..\\ACL_9th-F1\\F1\\'

circuits              = pd.read_csv(path + 'circuits.csv')
constructors          = pd.read_csv(path + 'constructors.csv')
constructor_results   = pd.read_csv(path + 'constructor_results.csv')
constructor_standings = pd.read_csv(path + 'constructor_standings.csv')
drivers               = pd.read_csv(path + 'drivers.csv')
driver_standings      = pd.read_csv(path + 'driver_standings.csv')
races                 = pd.read_csv(path + 'races.csv')
results               = pd.read_csv(path + 'results.csv')
qualifying            = pd.read_csv(path + 'qualifying.csv')
lap_times             = pd.read_csv(path + 'lap_times.csv')
pit_stops             = pd.read_csv(path + 'pit_stops.csv')
sprint_results        = pd.read_csv(path + 'sprint_results.csv')
seasons               = pd.read_csv(path + 'seasons.csv')
status                = pd.read_csv(path + 'status.csv')

## 1.3 Check table sizes
Shape = (rows, columns).

In [ ]:

print("lap_times:", lap_times.shape)
print("pit_stops:", pit_stops.shape)
print("sprint_results:", sprint_results.shape)
print("seasons:", seasons.shape)
print("status:", status.shape)

**Observations (from the shapes above):**
- All 14 tables loaded. Sizes range from 75 rows (`seasons`) to 589,081 rows (`lap_times`).
- `results` has 26,759 rows while `qualifying` has only 10,494, so qualifying has far fewer records.

**To check in the EDA:**
- Which years `qualifying` covers, and why it has fewer rows than `results`.
- Whether `results` has exactly one row per driver per race (the brief's required grain).
- Which years the data covers (`seasons` has 75 rows; confirm the year range).

# 2. Table-by-Table Audit & Cleaning
Each table is inspected before any cleaning: structure, missing values, `\N` placeholders,
primary key validity, and anything unusual. Every observation is based on the output above it.

## 2.1 Races
One row = one race. This table gives each race's season, round and date, used for the time-based split
and for finding each race's previous race.
### 2.1.1 First look

In [ ]:
races.head()

In [ ]:
races.info()

In [ ]:
races.isnull().sum()

In [ ]:
(races == '\\N').sum()

### Number of races per season

In [ ]:
races['year'].value_counts().sort_index()

In [ ]:
print("raceId unique:", races['raceId'].is_unique)
print("raceId nulls:", races['raceId'].isnull().sum())

In [ ]:
races['year'].value_counts().sort_index().plot(kind='bar', figsize=(16, 4))
plt.title('Number of races per season')
plt.xlabel('Season')
plt.ylabel('Races')
plt.show()

In [ ]:
races.sort_values('date')[['raceId', 'year', 'round', 'name', 'date']].head()

In [ ]:
races_sorted = races.sort_values(['year', 'date'])
races_sorted['round_by_date'] = races_sorted.groupby('year').cumcount() + 1
(races_sorted['round'] != races_sorted['round_by_date']).sum()

In [ ]:
races['date'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()

### Observations (races)

**Size and keys**
- 1,125 races and 18 columns (`races.info()`).
- `raceId` is unique and non-null, so it is a valid primary key (2.1.3).

**Missing values**
- `.isnull()` reports 0 missing values in every column, but the `\N` count shows many hidden ones:
  `time` (731 of 1,125), practice and qualifying session dates/times (1,035–1,072) and sprint
  dates/times (1,107–1,110). `.isnull()` alone does not detect this dataset's missing values.

**Coverage**
- 75 seasons, 1950–2024, with no year missing (`Length: 75`).
- Season length grew from 7 races (1950) to 24 (2024); 2020 had only 17 races (2.1.4 plot).

**Ordering**
- `raceId` is not in time order: the earliest race (1950 British GP, 13 May 1950) has `raceId` 833,
  while `raceId` 1 is the 2009 Australian GP. `raceId` therefore cannot be used to find a race's
  previous race.
- Within every season, `round` matches date order (0 mismatches), so `year` + `round` gives a
  reliable time order.
- All dates follow the `YYYY-MM-DD` format (check returned `True`), but they are stored as text
  (`object`) and must be converted to a date type.   ← keep only if cell 15 printed True

**Content**
- The Indianapolis 500 is listed as a championship race (round 3 of 1950). Whether to keep it is
  decided in Section 3, since it affects several tables.

### Cleaning
Based on the observations above:
- `\N` is replaced with NaN so pandas recognises missing values.
- `date` is converted from text to a date type.
- `name` is renamed to `race_name`, since `circuits` and `constructors` also have a `name` column,
  which would clash when tables are joined.
- Dropped columns:
  - `url`: Wikipedia link, no race information.
  - `time`, practice, qualifying and sprint date/time columns: when sessions took place, not their results;
    mostly `\N` (731–1,110 of 1,125).
All changes are made on a copy; the raw `races` table is unchanged.

In [ ]:
races_clean = races.copy()
races_clean = races_clean.replace('\\N', np.nan)
races_clean['date'] = pd.to_datetime(races_clean['date'], format='%Y-%m-%d')
races_clean = races_clean.rename(columns={'name': 'race_name'})
races_clean = races_clean.drop(columns=['url', 'time',
                                        'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time',
                                        'fp3_date', 'fp3_time', 'quali_date', 'quali_time',
                                        'sprint_date', 'sprint_time'])

### After cleaning

In [ ]:
races_clean.head()

In [ ]:
races_clean.info()

In [ ]:
(races_clean == '\\N').sum().sum()

In [ ]:
races_clean[['year', 'round']].describe()

### Indianapolis 500 races
Identified here for the keep/exclude decision in Section 3.

In [ ]:
indy_ids = races_clean[races_clean['race_name'] == 'Indianapolis 500']['raceId']
races_clean[races_clean['raceId'].isin(indy_ids)][['raceId', 'year', 'round', 'race_name']]

### 2.1.10 Summary (races)

**Findings (EDA)**
- 1,125 races over 75 seasons (1950–2024); `raceId` is a valid primary key.
- Missing values are hidden as `\N` (not detected by `.isnull()`), concentrated in session schedule columns.
- `raceId` is not chronological; `year` + `round` gives the correct time order (verified: 0 mismatches with date order).
- Dates were stored as text, all in `YYYY-MM-DD` format.
- The Indianapolis 500 appears as a championship race 11 times, once per season from 1950 to 1960
  (verified in the data), matching the F1 background document.

**Cleaning applied**
- `\N` → NaN; `date` converted to datetime; `name` renamed to `race_name`.
- 12 columns dropped (URL and session schedule columns, mostly `\N`).
- Result: `races_clean` with 6 columns, 1,125 rows, no missing values, no `\N`, valid year/round ranges.

**Rules for the rest of the notebook**
- Races are always ordered by `year` + `round`, never by `raceId`.
- Use `races_clean` (not `races`) in all later joins; the race name column is `race_name`.

**Deferred to Section 3**
- Keep or exclude the Indianapolis 500 (its 11 `raceId`s are saved in `indy_ids`); this affects
  `results`, `qualifying` and other tables.
- Foreign key check: every `races.circuitId` must exist in `circuits.circuitId`.

## 2.2 Circuits 
this table provides the information about the circuits including name , location geographically and in country
### first look

In [ ]:
circuits.head()

In [ ]:
circuits.info()

In [ ]:
circuits.describe()

In [ ]:
(circuits == '\\N').sum()

In [ ]:
circuits.duplicated().sum()

### Notes 
in the following cell we were looking at the values of the countries and we noticed that there has been duplication (not explicitily) but the value usa and united stated that should have been considered same value 

In [ ]:
circuits['country'].value_counts()

In [ ]:
print("circuitId unique:", circuits['circuitId'].is_unique)
print("circuitId nulls:", circuits['circuitId'].isnull().sum())

**Observations (circuits):**
- 77 circuits, no missing values and no `\N` sentinels.
- `circuitId` is unique and non-null, so it is a valid primary key (IDs go up to 80, so some numbers are unused).
- `alt` minimum is -7 m: a circuit below sea level, a real value rather than an error.
- **Inconsistent category:** the US appears as both `USA` (11) and `United States` (1).
  This must be standardised in cleaning, otherwise Question 2 (home races) would treat them as different countries.
- Columns we will use: `circuitId` (join key), `country` (Question 2).
- Columns that are irrelevant:CircuitRef, Name and url
- Other columns **may** be used in feature engineering as pasr races info.
- There are now row, circuitId,circuitRef or name Duplicates.

### Cleaning circuits table 
1. Fix duplicated USA - United States
2. Drop Unwanted Column in Circuits table

In [ ]:
# fix duplication cell 
circuits_clean = circuits.copy()
circuits_clean['country'] = circuits_clean['country'].replace({'United States': 'USA'})

In [ ]:
circuits_clean['country'].value_counts()

### Analysis of the coloumns 
1. url --> cannot be used in either training nor in data analysis because it doesnot hold information (the url string itself )
2. circuit ref , circuit name , location and country --> cannot be used in training as they are strings but would be useful in answering the data analysis questions
3. Rest of the coloumns --> numerical coloumns that could be used in the training but we refrain from dropping now as to see their coleration 

In [ ]:
circuits_clean.drop(columns=['url',], inplace=True)

In [ ]:
circuits_clean.head()

## 2.3 Constructor_results 
this table is mainly about constructors and their perfromances respectively in race (including points , status and race itself 
### Taking a look

In [ ]:
constructor_results.head()

In [ ]:
constructor_results.info()

In [ ]:
constructor_results.isnull().sum()

In [ ]:
(constructor_results == '\\N').sum()

### Notes
we notice by running the below 2 cells that there are only **17 values of D** in the status column and we research that these were the year Mclaren was disqualified from results but the races points were still documented so we will reject the disqualification matter and deal with the actual results and will later change the positions from E to the valid position

- also we noticed that all of the constructors here are not classified meaning they are not on podium 

In [ ]:
constructor_results['status'].value_counts()

In [ ]:
constructor_results[constructor_results['status'] != '\\N']

In [ ]:
import matplotlib.pyplot as plt

# Plot using matplotlib by passing the dataframe column
plt.hist(constructor_results['points'],bins=55, color='skyblue', edgecolor='black')

plt.title('points of the constrcutors ')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()


In [ ]:
constructor_results['points'].value_counts()

### Notes 
a lot of constrcutors (more than 6985 one ) got zero points on race ( because any one whos position from 11 to 20 got 0 points )
### Clean up 
1. changing /n values 
2. renaming colomn points

In [ ]:
constructor_results_clean = constructor_results.copy()
constructor_results_clean['status'] = constructor_results_clean['status'].replace('\\N', np.nan)
constructor_results_clean = constructor_results_clean.rename(columns={'points': 'race_specific_contsructor_points'})

In [ ]:
constructor_results_clean.head()

## 2.4 Constructor standings 
this table also is centered around the constructors . it holds information about the cummlative points (included the race) of each constructor and race and their position 
### take a look

In [ ]:
constructor_standings.head()

In [ ]:
constructor_standings.info()

In [ ]:
constructor_standings.describe()

In [ ]:
constructor_standings.isnull().sum()

In [ ]:
(constructor_standings == '\\N').sum()

### Notes 
- no null values of \n in this table 
- in the next cell we looked at position text and we saw only one **E** which means excluded (we have to delete those rows later)


In [ ]:
constructor_standings['positionText'].value_counts()


### Notes 
in the following cell we also checked for wether in the same race they have duplicated on same position which they not appear to be 

In [ ]:
duplicate_rows = constructor_standings[constructor_standings.duplicated(subset=['raceId', 'position'], keep=False)]
duplicate_rows

### Data cleaning 
- renaming the follwoing coloumns { points , position , positionText, wins}
- position text would not be used in training but it would be useful in extractign data and data analysis 



In [ ]:
constructor_standings_clean = constructor_standings.copy()
constructor_standings_clean = constructor_standings_clean.rename(columns={
    "points": "points_cumm_raceIncl",
    "position": "team_position_soFar_inConstrChamp",
    "positionText": "same_as_team_postion_but_with_E_for_2007_crisis",
    "wins": "constructor_wins"
})

In [ ]:
constructor_standings_clean.info()

## 2.5 Constructors 
this tables introduces the comstructors and their nationalities 
### Take a look

In [ ]:
constructors.info()

In [ ]:
(constructors == '\\N').sum()

In [ ]:
constructors['nationality'].value_counts()

In [ ]:
print("constructorId unique:", constructors['constructorId'].is_unique)
print("constructorId nulls:", constructors['constructorId'].isnull().sum())

In [ ]:
constructors[constructors['name'].duplicated(keep=False)]

### Notes 
- no constructors are duplicated 
- nationalities is duplicated in a sense which is east german and germany due to the split that happened to them between 1949 and 1990
- **So we notice from running the cell below that different constructors can have same nationality so we will leave this column as it means it may be that some nationalities perform better as for constructors**

In [ ]:
constructors[constructors['nationality']=='British']

### cleaning and observations about the coloumns
#### oberservation 
- url --> would be dropped immediatly
- constructorRef,name and nationality --> wouldnt be used in training as they are objects but would be useful in data analysis
- would say that about the whole table
#### Cleaning 
- renaming {nationality}
- changing east german to german in order to unify 

In [ ]:
constructors_clean = constructors.copy()
constructors_clean = constructors_clean.rename(columns={'nationality': 'constructor_nationality'})
constructors_clean['constructor_nationality'] = constructors_clean['constructor_nationality'].replace({'East German': 'German'})
constructors_clean = constructors_clean.drop(columns=['url', 'constructorRef'])

In [ ]:
constructors_clean['constructor_nationality'].value_counts()

In [ ]:
constructors_clean.info()

## 2.6 Driver_standings 
this tables shows the perfromance of the driver in races across the years with their resepctive accumplated points including the race and their psoition after the race 
### take a look

In [ ]:
driver_standings.info()

In [ ]:
driver_standings['positionText'].value_counts()

In [ ]:
(driver_standings == '\\N').sum()

### Notes 
- We observe that there is one row with position text = d dueto the incident of michael schumacher . this will be resolved later
- also there is no \n values in this table 

In [ ]:
driver_standings[driver_standings['positionText'] == 'D']

### cleaning and observations 
#### observations 
- position Text would be used as data analysis
#### cleaning 
- renaming the following coloumns {points , position, positionText, wins} for easier intreptations 


In [ ]:
driver_standings_clean = driver_standings.copy()
driver_standings_clean = driver_standings_clean.rename(columns={
    'points': 'driver_points_cumm',
    'position': 'driver_position1-in-rankings-of-season-so-far',
    'wins': 'driver_wins_cumm',
    'positionText': 'driver_position_sofar_text'
})

In [ ]:
driver_standings_clean.info()

## 2.7 Drivers 
this tables witholds information about teh drivers and their nationalties and Date of birth 
#### take a look

In [ ]:
drivers.head()

In [ ]:
drivers.info()

In [ ]:
drivers.duplicated().sum()
#nothing is duplicated
has_duplicates = drivers['driverId'].duplicated().any()
print(f"Are there duplicates? {has_duplicates}")

In [ ]:
(drivers == '\\N').sum()

### Cleaning and observations 
#### Observations 
- url --> be droppped ( same reason as prev)
- number --> would be dropped as 802 out of 861 of the coloumn is \n
- code  --> would be dropped as 757 out of 861 of the coloumn is \n
- other object (surname , forname & ref ) help in recognizing the driver not the training
- date of birth would be changed numerical in order to use to for feature coleration
#### cleaning 
- dropping coloumn { url,code , number ,driverref}
- merging forename and surname (for data anaylsis purposes )
- chaning dob into numerical format
- renaming the nationality 

In [ ]:
from datetime import datetime

drivers_clean = drivers.copy()
drivers_clean = drivers_clean.rename(columns={'nationality': 'drivers_nationality'})
drivers_clean = drivers_clean.drop(columns=['number', 'code', 'url', 'driverRef'])
# Merge name columns with a space between them
drivers_clean['driver_name'] = drivers_clean['forename'] + ' ' + drivers_clean['surname']
drivers_clean = drivers_clean.drop(columns=['forename', 'surname'])

today = datetime.now()
drivers_clean['dob'] = pd.to_datetime(drivers_clean['dob'])
drivers_clean['age'] = ((today - drivers_clean['dob']).dt.days / 365.25).astype(int)
drivers_clean = drivers_clean.drop(columns=['dob'])

In [ ]:
drivers_clean.head()

In [ ]:
drivers_clean.info()

## 2.8 Lap times
this table have analysis of drivers for each lap they have done per race 
### take a look


In [ ]:
lap_times.info()

In [ ]:
(lap_times == '\\N').sum()
lap_times.isnull().sum()

In [ ]:
duplicate_rows = lap_times[lap_times.duplicated(subset=['raceId', 'driverId','lap'], keep=False)]
duplicate_rows

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Count the laps (rows) per driver in your target race (e.g., raceId 841)
target_race_id = 841
race_laps = lap_times[lap_times["raceId"] == target_race_id]
lap_counts = (
    race_laps.groupby("driverId").size().reset_index(name="laps_completed")
)

# 2. Merge with your clean drivers table to get their names
final_df = pd.merge(lap_counts, drivers_clean, on="driverId", how="left")
final_df = final_df.sort_values(by="laps_completed", ascending=False)

# 3. Create a clean, simple bar plot
plt.figure(figsize=(12, 5))

# Fixed the deprecation warning by adding hue= and legend=False
ax = sns.barplot(
    data=final_df,
    x="driver_name",
    y="laps_completed",
    hue="driver_name",
    legend=False,
    palette="viridis",
)

# 4. Add the labels on top of the bars (Fixed the list error here)
ax.bar_label(ax.containers[0], fontsize=9)

# 5. Clean layout adjustments
plt.title(f"Laps Completed in Race {target_race_id}", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Driver")
plt.ylabel("Total Laps")
plt.tight_layout()

plt.show()


###  Key Observations: Race Lap Completion Profile (Race ID: 841)

> **Context:** This analysis isolates a single Grand Prix event from the `lap_times` table to evaluate driver reliability and completion rates. 

* **The Lead Pack Plateau:** The dominant, flat horizontal plateau on the left side of our visualization represents the drivers who completed the maximum race distance of **58 laps**. This confirms a clean run without race-ending incidents or terminal mechanical failures.
* **Isolating DNFs (Did Not Finish):** The stepping-down distribution on the right side of the chart visually identifies mid-race retirements. 
* **Reliability Metrics:** Drivers with drastically shorter bars represent early-race casualties (e.g., first-lap collisions), while those closer to the 58-lap threshold indicate late-stage heartbreaks, such as mechanical failures or retirements in the closing stages of the Grand Prix.

---

#### 🛠️ Data Architecture Note
* **`lap_times` Integrity:** Verified. The counting script aggregates unique row entries matching the compound signature `['raceId', 'driverId']`. 
* **Data Completeness:** The maximum value matches the total official race lap count, ensuring no overlapping lap time anomalies exist in this segment of the database.


### Cleaning and Observations
#### Observations 
- time would be dropped as it is object and there is its numerical correspondant
- every other feature would be used later in feature engineering sections
#### Cleaning 
- drop time
- rename {position,milliseconds}

In [ ]:
lap_times['position'].value_counts()

In [ ]:
lap_times_clean = lap_times.copy()
lap_times_clean = lap_times_clean.drop(columns=['time'])
lap_times_clean = lap_times_clean.rename(columns={
    'position': 'position_in_lap',
    'milliseconds': 'milliseconds_per_lap'
})

In [ ]:
lap_times_clean.head()

## 2.9 pit_stops 
this table provides information  when cars entered the pits, how long they stayed, and how it affected their race strategy


In [ ]:
pit_stops.head()

In [ ]:
pit_stops.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df_pit_stops = pd.DataFrame(pit_stops) is already loaded

# 1. AGGREGATE PER RACE: Include 'raceId' in the groupby
# This finds the maximum pit stop number for each driver in *each separate race*
df_pit_stops = pd.DataFrame(pit_stops)
driver_race_totals = df_pit_stops.groupby(["raceId", "driverId"])["stop"].max().reset_index()
driver_race_totals.rename(columns={"stop": "Total_Pit_Stops"}, inplace=True)

# -------------------------------------------------------------
# OPTION A: Filter to look at ONE specific race (e.g., raceId 1110)
# This fixes your data so numbers like 51, 52, or 70 drop down to standard 1, 2, or 3 stops.
target_race_id = 1110  
single_race_df = driver_race_totals[driver_race_totals["raceId"] == target_race_id]

# Calculate the frequency bucket for just this one race
frequency_df = single_race_df["Total_Pit_Stops"].value_counts().reset_index()
frequency_df.columns = ["Strategy (Num of Stops)", "Driver Count"]
frequency_df.sort_values(by="Strategy (Num of Stops)", inplace=True)
# -------------------------------------------------------------

# -------------------------------------------------------------
# OPTION B: Un-comment the 3 lines below if you want an overall historical average 
# of how frequent 1, 2, or 3 stop strategies are across ALL individual races combined:
# frequency_df = driver_race_totals["Total_Pit_Stops"].value_counts().reset_index()
# frequency_df.columns = ["Strategy (Num of Stops)", "Driver Count"]
# frequency_df.sort_values(by="Strategy (Num of Stops)", inplace=True)
# -------------------------------------------------------------

print("--- Strategy Frequency Distribution ---")
print(frequency_df.to_string(index=False))

# 2. VISUALIZE: Plot the strategy frequencies 
plt.figure(figsize=(7, 4.5))
sns.set_theme(style="whitegrid")

ax = sns.barplot(x="Strategy (Num of Stops)", y="Driver Count", data=frequency_df, color="crimson")

# Formatting 
plt.title(f"F1 Pit Stop Strategy Frequency (Race ID: {target_race_id})", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Number of Pit Stops Taken", fontsize=11)
plt.ylabel("Number of Drivers", fontsize=11)
plt.yticks(range(0, frequency_df["Driver Count"].max() + 2))

# Add value labels over the bars
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{int(p.get_height())}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='baseline', fontsize=11, color='black', 
                    xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df_pit_stops = pd.DataFrame(pit_stops) is already loaded

# 2. AGGREGATE: Group by driver to extract the maximum value in the 'stop' column
driver_totals = df_pit_stops.groupby(["driverId"])["stop"].max().reset_index()

# FIX: Changed "stops" to "stop" to match your actual column name
driver_totals.rename(columns={"stop": "Total_Pit_Stops"}, inplace=True)

print("--- Total Pit Stops Per Driver ---")
#print(driver_totals.to_string(index=False))

# 3. FREQUENCY DISTRIBUTION: Count how many drivers fall into each strategy bucket
frequency_df = driver_totals["Total_Pit_Stops"].value_counts().reset_index()
frequency_df.columns = ["Strategy (Num of Stops)", "Driver Count"]
frequency_df.sort_values(by="Strategy (Num of Stops)", inplace=True)

print("\n--- Strategy Frequency Distribution ---")
#print(frequency_df.to_string(index=False))

# 4. VISUALIZE: Plot the strategy frequencies 
plt.figure(figsize=(10, 5)) # Slightly wider layout to accommodate historical data ranges
sns.set_theme(style="whitegrid")

# Create a bar plot of the aggregated frequencies
ax = sns.barplot(x="Strategy (Num of Stops)", y="Driver Count", data=frequency_df, color="crimson")

# Formatting 
plt.title("F1 Pit Stop Strategy Frequency Distribution", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Number of Pit Stops Taken", fontsize=11)
plt.ylabel("Number of Drivers", fontsize=11)
plt.yticks(range(0, frequency_df["Driver Count"].max() + 2))

# Add value labels over the bars
for p in ax.patches:
    if p.get_height() > 0: # Only label bars that have data
        ax.annotate(f'{int(p.get_height())}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='baseline', fontsize=10, color='black', 
                    xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()


### Cleaning and Observations 
#### observations 
-  time --> would be dropped as it indicated the time that the pitstop
#### cleaning 
-  duration --> needs to be changed into numerical
-  renaming coloumns { lap , milliseconds }


In [ ]:
# 1. Define a function to convert "MM:SS.fff" or "SS.fff" string formats into total seconds
def time_to_seconds(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    # If there is a colon (MM:SS.fff), split into minutes and seconds
    if ":" in val:
        try:
            minutes, seconds = val.split(":")
            return (float(minutes) * 60) + float(seconds)
        except ValueError:
            return None # Handles any unexpected bad string formatting
        
    # If it's already just seconds (SS.fff), convert directly to float
    try:
        return float(val)
    except ValueError:
        return None

pit_stops_clean = pit_stops.copy()
pit_stops_clean = pit_stops_clean.drop(columns=['time'])
pit_stops_clean = pit_stops_clean.rename(columns={
    'lap': 'pitstop_lap',
    'milliseconds': 'pitstop_timetaken_in_ml'
})
# Convert duration from text to seconds (float)
pit_stops_clean['duration'] = pit_stops_clean['duration'].apply(time_to_seconds)
pit_stops_clean['duration'] = pd.to_numeric(pit_stops_clean['duration'], errors='coerce')

In [ ]:
(pit_stops_clean == '\\N').sum()

In [ ]:
pit_stops_clean.isnull().sum()

In [ ]:
pit_stops_clean.head()

In [ ]:
pit_stops_clean.info()

### 2.10 Qualifying 
this table talks about the qualifying rounds that the drivers enter before the race start in order to decide their position on the grid 
- Position --> driver's race position
- q1 All drivers
- q2 top 15
- q3 top 10
#### take a look

In [ ]:
qualifying_clean = qualifying.copy()

def to_seconds(t):
    if pd.isna(t) or t == '\\N':
        return '\\N'                   
    m, s = t.split(':')
    return int(m) * 60 + float(s)

for c in ['q1', 'q2', 'q3']:
    qualifying_clean[c + '_s'] = qualifying_clean[c].map(to_seconds)
    
qualifying_clean.drop(columns=['q1', 'q2', 'q3'], inplace=True)
qualifying_clean.info()

In [ ]:
qualifying_clean.info()

In [ ]:
qualifying_clean['qualifyId'].duplicated().sum()

In [ ]:
qualifying_clean.duplicated(subset=['raceId', 'driverId']).sum()

checking for duplicated raceid and driverid and car number.

In [ ]:
qualifying_clean.duplicated(subset=['raceId', 'driverId','number']).sum()

There are null values in q2 and q3 other than the /N

In [ ]:
qualifying_clean.isna().sum()

There are many /N values in Q1, Q2 and Q3 where /N in them means they didnt qualify for this round 

In [ ]:
has_backslash_n = (qualifying_clean == r'\N').any().any()
print(f"Does \\N exist in the dataset? {has_backslash_n}")

for col in qualifying_clean.columns:
    count = (qualifying_clean[col] == r'\N').sum()
    if count > 0:
        print(f"Column '{col}' has {count} rows with '\\N'")

In [ ]:
qualifying_clean=qualifying_clean.replace(r'\N', np.nan, inplace=True)

In [ ]:
qualifying_clean.isna().sum()

In [ ]:
qualifying_clean.rename(columns={"position": "qualifying_position"}, inplace=True)

In [ ]:
duplicate_rows = qualifying_clean[qualifying_clean.duplicated(subset=['raceId', 'driverId'], keep=False)]

In [ ]:
qualifying_clean.head()

Sprint Results

In [ ]:
sprint_results_clean = sprint_results.copy()

In [ ]:
cols_with_N = [col for col in sprint_results_clean.columns
               if (sprint_results_clean[col] == r'\N').any()]
print(f"Columns with '\\N': {cols_with_N}")


In [ ]:
for col in sprint_results.columns:
    count = (sprint_results[col] == r'\N').sum()
    if count > 0:
        print(f"Column '{col}' has {count} rows with '\\N'")

In [ ]:
print("resultId duplicates:", sprint_results['resultId'].duplicated().sum())
print("Same driver twice in one sprint:", sprint_results.duplicated(subset=['raceId', 'driverId']).sum())
print("Broken raceId:", (~sprint_results['raceId'].isin(races['raceId'])).sum())
print("Broken driverId:", (~sprint_results['driverId'].isin(drivers['driverId'])).sum())
print("Broken constructorId:", (~sprint_results['constructorId'].isin(constructors['constructorId'])).sum())
print("Broken statusId:", (~sprint_results['statusId'].isin(status['statusId'])).sum())
sprint_results.rename(columns={"positionOrder": "sprint_position"}, inplace=True)
sprint_years = sprint_results.merge(races[['raceId', 'year']], on='raceId')
print(sprint_years.groupby('year')['raceId'].nunique())
sprint_results_clean = sprint_results_clean.drop(columns = ['position', 'time', 'laps', 'fastestLap'])

Notes :The 20 rows with \N are drivers who did not finish. Those who completed laps before retiring have a fastest lap but no finishing time. Those who retired on lap 1 have no lap data at all. Some non-finishers are still classified (have a position) under the ~90%-distance rule, but 2024 sprints classify non-finishers inconsistently. For this reason position was dropped in favour of positionOrder + statusId.

In [ ]:
rows_with_N = sprint_results.loc[
    (sprint_results[cols_with_N] == r'\N').any(axis=1),
    ['raceId', 'driverId', 'statusId'] + cols_with_N
]

print(rows_with_N.shape)
rows_with_N
rows_with_N['statusId'].value_counts()

## 2.2 Results
One row is expected to be one driver in one race (verified below). This table holds the target
(`positionOrder`) and the starting grid position.

### 2.2.1 First look

In [ ]:
results.head()

**Note:** Each row is one driver in one race (same `raceId`, different drivers). The `time` column mixes two
formats: the winner's total race time (`1:34:50.616`) and other drivers' gap to the winner (`+5.478`),
so it is not one consistent measurement.

In [ ]:
results.info()

**Note:** 26,759 rows, 18 columns, and `info()` reports no missing values. However, 7 columns that should
be numeric are stored as text (`object`): `number`, `position`, `milliseconds`, `fastestLap`, `rank`,
`fastestLapTime`, `fastestLapSpeed`. This suggests non-numeric placeholders inside (checked in 2.2.2).

In [ ]:
results.describe()

**Note:** Unusual values to investigate:
- `grid` minimum is 0 (not a real grid slot; meaning checked in 2.2.4).
- `points` maximum is 50 and `laps` maximum is 200, both far above typical values (checked in 2.2.4).
- `positionOrder` goes up to 39, so some races had up to 39 entrants.
- Median `points` is 0: most drivers score nothing in a given race.

### 2.2.2 Missing values

In [ ]:
results.isnull().sum()

In [ ]:
(results == '\\N').sum()

### 2.2.3 Keys and grain

In [ ]:
print("resultId unique:", results['resultId'].is_unique)
print("resultId nulls:", results['resultId'].isnull().sum())

In [ ]:
results.duplicated(subset=['raceId', 'driverId']).sum()

**Note:** 91 rows repeat a (`raceId`, `driverId`) pair, so the required grain of one row per driver per race
is broken. Their origin (expected: early shared drives) is checked next.

### 2.2.4 Unusual values from describe()

In [ ]:
print("rows with grid = 0:", (results['grid'] == 0).sum())

**Note:** 1,638 rows have `grid` = 0. This is not a real starting slot; its meaning (pit-lane start or
missing grid data) is checked by season next.

In [ ]:
results[results['points'] == results['points'].max()].merge(
    races_clean[['raceId', 'year', 'race_name']], on='raceId')

**Note:** The only row with 50 points is the winner of the 2014 Abu Dhabi Grand Prix. Whether that race
awarded double points is checked next by looking at the other finishers' points.

In [ ]:
results[results['laps'] == results['laps'].max()].merge(
    races_clean[['raceId', 'year', 'race_name']], on='raceId')

**Note:** All 123 rows with 200 laps are from the Indianapolis 500 (across several 1950s seasons), so the
200-lap distance is specific to that race, one more way it differs from a normal Grand Prix.
Some rows also show several drivers with the same car number, team and finishing result in the same
race (e.g. three drivers on car 45 in the 1954 Indianapolis 500), meaning one car's result was
recorded for multiple drivers.

### 2.2.5 Duplicate driver rows (grain check)
Which seasons do the 91 repeated (`raceId`, `driverId`) rows come from, and what do they look like?

In [ ]:
dups = results[results.duplicated(subset=['raceId', 'driverId'], keep=False)]
dups = dups.merge(races_clean[['raceId', 'year', 'race_name']], on='raceId')
dups['year'].value_counts().sort_index()

### One example of a duplicate:

In [ ]:
dups.sort_values(['raceId', 'driverId'])[['year', 'race_name', 'driverId', 'constructorId',
                                          'number', 'grid', 'positionOrder', 'points', 'laps']].head(10)

In [ ]:
dups.groupby(['raceId', 'driverId']).size().value_counts()

In [ ]:
triples = dups.groupby(['raceId', 'driverId']).filter(lambda g: len(g) == 3)
triples.sort_values(['raceId', 'driverId'])[['year', 'race_name','raceId', 'driverId', 'constructorId',
                                             'number', 'grid', 'positionOrder', 'points', 'laps']]

In [ ]:
driver_info = driver_standings[(driver_standings['driverId'] == 521) & (driver_standings['raceId'] == 809)]
driver_info

**Note (drivers listed three times):** All 6 cases are from the 1953 and 1954 Indianapolis 500 and the
1955 Argentine GP. Drivers drove parts of the race in several cars, and cars were shared by several drivers
(e.g. car 28 in the 1955 Argentine GP appears for drivers 501 and 554 with the same result). Points were
split between drivers sharing a car (e.g. 1.33 points). In the Indianapolis 500 a driver could even appear
for cars of different teams. In all 6 cases, the row with the best `positionOrder` is also the car in
which the driver completed the most laps.

**Note (duplicates):** The grain check found 91 extra rows repeating a (`raceId`, `driverId`) pair.
These belong to 85 driver-race groups: 79 drivers are listed twice and 6 are listed three times in the same
race (79 × 2 + 6 × 3 = 176 rows in total, of which 91 are extra copies).

The repeats are concentrated in 1950–1957 (peak 1953–1956), with a few up to 1964 and two in 1978.
The examples show two causes:
- **Shared drives:** a driver raced two cars of the same team in one race
  (e.g. 1964 US GP, driver 373, cars 1 and 2, finishing 7th and 12th).
- **Multiple non-starting entries:** a driver entered with two different teams but did not start
  (e.g. 1962 British GP and 1978 Italian GP: `grid` 0, 0 laps in both rows).

A rule to keep exactly one row per driver per race is chosen in cleaning.

### 2.2.6 Grid = 0 by season
Is `grid` = 0 spread across all eras (pit-lane starts) or concentrated in certain years (missing grid data)?

In [ ]:
results[results['grid'] == 0].merge(races_clean[['raceId', 'year']], on='raceId')['year'] \
    .value_counts().sort_index().plot(kind='bar', figsize=(16, 4))
plt.title('Rows with grid = 0 per season')
plt.show()

**Note:** `grid` = 0 is concentrated in 1977–1983 and 1988–1992 (peak 1989, ~200 rows), the eras with more
entries than grid places, and is rare after 1995 (0–18 per season). This suggests `grid` = 0 mostly marks
drivers who did not start in the older eras, and pit-lane starts in the modern era (checked below).

### 2.2.7 Points in the 2014 Abu Dhabi Grand Prix
Checking whether the 50-point maximum comes from a race that awarded double points.

In [ ]:
results[results['raceId'] == 918][['positionOrder', 'points']].sort_values('positionOrder').head(5)

**Note:** The top five in the 2014 Abu Dhabi GP scored 50, 36, 30, 24 and 20 points, exactly double the
normal 25, 18, 15, 12 and 10, so this race awarded double points. The 50-point maximum is real, not an
error. Points-based features would overweight this race; position-based features avoid this.

### 2.2.8 Rows with no official position
When `position` is `\N`, what does `positionText` record instead?

In [ ]:
results[results['position'] == '\\N']['positionText'].value_counts()

In [ ]:
results['positionText'].value_counts()

**Note:** Every row with `position` = `\N` has a reason code in `positionText`: R (retired, 8,897),
F (failed to qualify, 1,368), W (withdrawn, 336), N (not classified, 190), D (disqualified, 151),
E (excluded, 9). Two rows have a number instead (11 and 14) and are inspected below.
Rows with F or W never started the race; since this is known at the prediction cutoff, they are
candidates for exclusion from the modeling table.

### 2.2.9 The two odd rows

In [ ]:
results[(results['position'] == '\\N') & (results['positionText'].isin(['11', '14']))].merge(
    races_clean[['raceId', 'year', 'race_name']], on='raceId')

**Note:** The two rows with a numeric `positionText` but `position` = `\N` are both from the 2005 Australian GP
(drivers 18 and 11, 56 and 55 laps, `positionText` 11 and 14). `positionOrder` agrees with `positionText`
(11 and 14), so `position` appears to be missing by mistake for these two rows. This does not affect the
target, which is built from `positionOrder`.

### 2.2.10 Does grid = 0 match “failed to qualify”?

In [ ]:
pd.crosstab(results['grid'] == 0, results['positionText'].isin(['F', 'W']),
            rownames=['grid = 0'], colnames=['F or W'])

**Note:** Of the 1,638 rows with `grid` = 0, 1,551 (~95%) are drivers coded F (failed to qualify) or
W (withdrawn). In this dataset `grid` = 0 therefore mostly means the driver did not start, not a pit-lane
start. Only 87 rows have `grid` = 0 for drivers who took part; these are the likely pit-lane starts.
Using `grid` = 0 as a number would wrongly rank these drivers ahead of pole position, so it needs handling
in cleaning.

### 2.2.11 Non-starters and grid = 0
Checks whether `grid` reliably tells us who started the race.

In [ ]:
odd = results[(results['grid'] != 0) & (results['positionText'].isin(['F', 'W']))]
odd = odd.merge(races_clean[['raceId', 'year']], on='raceId')
pd.crosstab(odd['year'], odd['positionText'])

20 are F - 133 are W

## Part A: The 133 W (withdrawn)
### Cell A1: did they race?

In [ ]:
odd[odd['positionText'] == 'W']['laps'].value_counts()

In [ ]:
odd[(odd['positionText'] == 'W') & (odd['laps'] > 0)] \
    .merge(races_clean[['raceId', 'race_name']], on='raceId') \
    [['year', 'race_name', 'driverId', 'grid', 'laps', 'positionOrder', 'statusId']]

In [ ]:
status[status['statusId'] == 54]

### Cell A2: the 2005 spike

In [ ]:
odd[odd['year'] == 2005].merge(races_clean[['raceId', 'race_name']], on='raceId')['race_name'].value_counts()

**Note (W drivers with a grid number, 133 rows):**
- **128 completed 0 laps:** they were given a grid slot from qualifying but withdrew before the start.
  14 of them come from a single race, the 2005 United States Grand Prix, where most of the field withdrew.
- **5 completed laps (4–25):** all have `statusId` 54 ("Withdrew" in the `status` table). They started the race
  and were withdrawn during it (e.g. two cars in the 1952 Swiss GP, both after 24 laps).

**Decision:** W therefore does not always mean "did not start", so the `started` flag treats W as a non-start
only when the driver completed 0 laps. The 5 drivers who raced are kept as starters (`podium` = 0).
The 128 who withdrew before the start had a grid slot at the prediction cutoff (after qualifying), so their
withdrawal was not yet known at prediction time. They are kept in the modelling table with `podium` = 0,
the same way as drivers who crash or break down during the race. Removing them would use information from
after the cutoff to decide which rows the model sees.

## Part B: The 20 F (failed to qualify)
### Cell B1: compare grid with qualifying position 

In [ ]:
f_grid = results[(results['grid'] != 0) & (results['positionText'] == 'F')]
f_grid = f_grid.merge(races_clean[['raceId', 'year', 'race_name']], on='raceId')

quali_pos = qualifying[['raceId', 'driverId', 'position']].rename(columns={'position': 'quali_position'})

f_grid.merge(quali_pos, on=['raceId', 'driverId'], how='left') \
    [['year', 'race_name', 'driverId', 'grid', 'quali_position', 'laps', 'positionOrder']]

In [ ]:
print("F drivers:", len(f_grid), "| races:", f_grid['raceId'].nunique())

### Cell B2: does their grid continue right after the last starter?

In [ ]:
f_races = f_grid['raceId'].unique()
field = results[results['raceId'].isin(f_races)][['raceId', 'driverId', 'grid', 'positionText', 'laps']]
field = field.merge(races_clean[['raceId', 'year', 'race_name']], on='raceId')
print(field.sort_values(['raceId', 'grid'])[['year', 'race_name', 'grid', 'positionText', 'laps']]
      .groupby(['year', 'race_name']).tail(5).to_string())

### 2.2.12 When did the 87 likely pit-lane starts happen?

In [ ]:
pit_starts = results[(results['grid'] == 0) & (~results['positionText'].isin(['F', 'W']))]
pit_starts.merge(races_clean[['raceId', 'year']], on='raceId')['year'].value_counts().sort_index()

**Note:** Of the 87 rows with `grid` = 0 for drivers who took part, 73 (~84%) are from 2015–2024, consistent
with pit-lane starts in the modern era. The remaining 14 (1981–1997) are ambiguous. These drivers did start
the race, but from behind the whole grid, so `grid` = 0 must not be read as "ahead of pole".

In [ ]:
pit_starts.merge(races_clean[['raceId', 'year', 'race_name']], on='raceId') \
    .query('year < 2000')[['year', 'race_name', 'driverId', 'positionText', 'positionOrder', 'laps', 'statusId']]

### 2.2.10 Cleaning
1. `\N` is replaced with NaN; `milliseconds`, `rank`, `fastestLap`, `fastestLapSpeed` are converted to numbers and
   `fastestLapTime` from text (e.g. `1:27.452`) to seconds (`fastestLapTime_s`).
2. **Duplicates:** for each (`raceId`, `driverId`) we keep the row with the best (lowest) `positionOrder`.
   This always works (`positionOrder` is never missing), matched the car with the most laps in all checked cases,
   and if any of a driver's cars finished on the podium, the driver was on the podium.
   Limitation: the kept row's `constructorId`/`grid` belong to that car.
   This is done before any joins, so later joins on (`raceId`, `driverId`) are one-to-one.
3. **Target:** `podium` = 1 if `positionOrder` ≤ 3, else 0.
4. **Flags:**
   - `started`: False if F, W with 0 laps, or `grid` = 0 with 0 laps.
   - `dns_known_at_cutoff`: True if F or `grid` = 0 with 0 laps; these rows are excluded from the modelling table in
     Section 3. W drivers who withdrew after getting a grid slot are kept (not yet known at the cutoff).
   - `pit_lane_start`: drivers who started with `grid` = 0.
5. **Dropped columns:** `resultId` (row ID, key verified) and `number` (car number, an identifier) by common sense;
   `position` because it equals `positionOrder` wherever both exist and is 41% `\N`;
   `time` because it holds the same information as `milliseconds` (missing in exactly the same rows) but as text
   mixing two formats. `fastestLapTime` is replaced by `fastestLapTime_s`.
6. **Kept as feature candidates** (decided in feature engineering with coverage and correlation checks):
   laps, milliseconds and the fastest-lap columns.
7. **Renamed** post-race columns to avoid clashing with same-named columns in other tables
   (e.g. `points` and `positionText` also exist in the standings tables with a different meaning):
   - `positionOrder` → `race_position_order` (the brief's target `positionOrder ≤ 3` is our `podium` column)
   - `positionText` → `race_position_text`
   - `points` → `race_points`, `laps` → `race_laps`, `milliseconds` → `race_ms`
   - fastest-lap columns → `fastest_lap_number`, `fastest_lap_rank`, `fastest_lap_time_s`, `fastest_lap_speed`
   Keys, `grid` and `statusId` keep their original names.

In [ ]:
results_clean = results.copy()

# 1. Sentinels and types
results_clean = results_clean.replace('\\N', np.nan)
for col in ['milliseconds', 'rank', 'fastestLap', 'fastestLapSpeed']:
    results_clean[col] = pd.to_numeric(results_clean[col])

def lap_to_seconds(t):
    if pd.isna(t):
        return np.nan
    minutes, seconds = t.split(':')
    return int(minutes) * 60 + float(seconds)

results_clean['fastestLapTime_s'] = results_clean['fastestLapTime'].apply(lap_to_seconds)

# 2. Duplicates: keep the best positionOrder per driver per race
results_clean = results_clean.sort_values(['raceId', 'driverId', 'positionOrder']) \
                             .drop_duplicates(subset=['raceId', 'driverId'], keep='first')

# 3. Target
results_clean['podium'] = (results_clean['positionOrder'] <= 3).astype(int)

# 4. Flags
f_code   = results_clean['positionText'] == 'F'
w_0laps  = (results_clean['positionText'] == 'W') & (results_clean['laps'] == 0)
no_grid0 = (results_clean['grid'] == 0) & (results_clean['laps'] == 0)

results_clean['started'] = ~(f_code | w_0laps | no_grid0)
results_clean['dns_known_at_cutoff'] = f_code | no_grid0
results_clean['pit_lane_start'] = (results_clean['grid'] == 0) & results_clean['started']

# 5. Drop columns justified by common sense or evidence
results_clean = results_clean.drop(columns=['resultId', 'number', 'position', 'time', 'fastestLapTime'])

# 6. Rename post-race columns so they don't clash with other tables
results_clean = results_clean.rename(columns={
    'positionOrder': 'race_position_order',
    'positionText': 'race_position_text',
    'points': 'race_points',
    'laps': 'race_laps',
    'milliseconds': 'race_ms',
    'fastestLap': 'fastest_lap_number',
    'rank': 'fastest_lap_rank',
    'fastestLapTime_s': 'fastest_lap_time_s',
    'fastestLapSpeed': 'fastest_lap_speed'
})

### 2.2.11 After cleaning

In [ ]:
results_clean.head()

In [ ]:
results_clean.info()

In [ ]:
print("rows:", len(results_clean), "(expected 26,759 − 91 = 26,668)")
print("duplicate driver-race pairs:", results_clean.duplicated(subset=['raceId', 'driverId']).sum())
print("\\N left:", (results_clean == '\\N').sum().sum())

In [ ]:
print(results_clean['started'].value_counts(), "\n")
print(results_clean['dns_known_at_cutoff'].value_counts(), "\n")
print("pit-lane starts:", results_clean['pit_lane_start'].sum())

In [ ]:
results_clean['podium'].value_counts(normalize=True)

In [ ]:
results_clean = results_clean.reset_index(drop=True)

In [ ]:
results_clean.info()

**After cleaning (results):**
- 26,668 rows, no duplicate (`raceId`, `driverId`) pairs and no `\N` left: the required grain is met.
- `started` = False for 1,711 rows; `dns_known_at_cutoff` = True for 1,583. The difference (128) is exactly the
  W drivers who withdrew after receiving a grid slot, kept in the modelling table with `podium` = 0.
- 72 pit-lane starts: one of the 73 modern `grid` = 0 rows completed 0 laps and is treated as a non-starter
  (the lap-1 edge case noted in the limitations).
- `podium` = 1 for 12.7% of rows: the target is imbalanced, so accuracy would be misleading and PR-AUC/F1 are used.

In [ ]:
from IPython.display import display, HTML

# 1. Put all your DataFrame heads into a dictionary with their names as keys
dfs_to_show = {
    "circuits_clean": circuits_clean.head(),
    "constructor_results_clean": constructor_results_clean.head(),
    "constructor_standings_clean": constructor_standings_clean.head(),
    "constructors_clean": constructors_clean.head(),
    "pit_stops_clean": pit_stops_clean.head(),
    "qualifying_clean": qualifying_clean.head(),
    "results_clean": results_clean.head(),
    "lap_times_clean": lap_times_clean.head(),
    "seasons": seasons.head(),
    "sprint_results_clean": sprint_results_clean.head(),
    "status": status.head(),
    "driver_standings_clean": driver_standings_clean.head(),
    "drivers_clean": drivers_clean.head(),
    "races_clean": races_clean.head()
}

# 2. Build a scrollable horizontal container
html_output = '<div style="display: flex; gap: 30px; overflow-x: auto; padding-bottom: 15px;">'

for name, df_head in dfs_to_show.items():
    html_output += '<div style="flex: 0 0 auto;">'
    html_output += f'<h3 style="margin-bottom: 5px; color: #4A5568;">{name}</h3>'
    html_output += df_head.to_html()
    html_output += '</div>'

html_output += '</div>'

# 3. Render the output
display(HTML(html_output))


In [ ]:
races_clean.drop(columns=['race_name','date'], inplace=True)
races_clean.rename(columns={'year': 'race_year', 'round': 'race_round'}, inplace=True)
races_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'races_clean.csv', index=False)

In [ ]:
circuits_clean.drop(columns=['name','location','lat','lng','alt','circuitRef'], inplace=True)
circuits_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'circuits_clean.csv', index=False)


In [ ]:
constructor_results_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'constructor_results_clean.csv', index=False)
constructor_results_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'constructor_results_clean.csv', index=False)
constructors_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'constructors_clean.csv', index=False)

pit_stops_clean.drop(columns=['duration'], inplace=True)
pit_stops_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'pit_stops_clean.csv', index=False)







In [ ]:
qualifying_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'qualifying_clean_with_qs&pos.csv', index=False)


results_clean.rename(columns={'statusId': 'statusId_race'}, inplace=True)
results_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'results_clean.csv', index=False)

lap_times_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'lap_times_clean.csv', index=False)
seasons.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'seasons.csv', index=False)

In [ ]:
sprint_results_clean.rename(columns={'positionOrder': 'sprint_position','grid': 'sprint_grid','fastestLapTime': 'sprint_fastest_laptime','points':'sprint_points','milliseconds':'sprint_ms','positionText':
                               'sprint_position_text'}, inplace=True)

In [ ]:
sprint_results_clean=pd.merge(
    sprint_results_clean, 
    status, 
    on=['statusId'], 
    
)

In [ ]:
sprint_results_clean.head()

In [ ]:
sprint_results_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'sprint_results_clean.csv', index=False)

In [ ]:
driver_standings_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'driver_standings_clean.csv', index=False)

In [ ]:
drivers_clean.drop(columns=['driver_name'], inplace=True)

In [ ]:
drivers_clean.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'drivers_clean.csv', index=False)

checking which drivers in same race have diff grid and qualifying order

In [ ]:
results_subset = results_clean[['driverId', 'raceId', 'grid']]
qualifying_subset = qualifying_clean[['driverId', 'raceId', 'qualifying_position']]


# 2. Merge the tables on both driverId and raceId
# An 'outer' join ensures you see all combinations, even if data is missing in one table
combined_grid_qual = pd.merge(
    results_subset, 
    qualifying_subset, 
    on=['driverId', 'raceId'], 
    how='outer'
)

# 3. View the combined DataFrame
print(f"Total combinations found: {len(combined_grid_qual)}")
display(combined_grid_qual.head(20))

In [ ]:
clean_combinations = combined_grid_qual.dropna(subset=['grid', 'qualifying_position'])

# 2. Count the matches and mismatches
same_position = (clean_combinations['grid'] == clean_combinations['qualifying_position']).sum()
different_position = (clean_combinations['grid'] != clean_combinations['qualifying_position']).sum()
total_rows = len(clean_combinations)

# 3. Calculate percentages
pct_same = (same_position / total_rows) * 100
pct_diff = (different_position / total_rows) * 100

# 4. Print the final counts
print(f"Total valid rows analyzed: {total_rows:,}\n")
print(f"Same Position (No Grid Penalties): {same_position:,} ({pct_same:.2f}%)")
print(f"Different Position (Grid Penalties/Changes): {different_position:,} ({pct_diff:.2f}%)")

In [ ]:
import pandas as pd

# 1. Merge tables to see who is in qualifying but might be missing from results
# We look at driverId and raceId across both clean datasets
failed_to_qualify_and_missing_from_results = pd.merge(
    qualifying_clean[['driverId', 'raceId']], 
    results_clean, 
    on=['driverId', 'raceId'], 
    how='left'
)

# 2. Find rows where the merge failed to find a matching race result 
# (if 'grid' is NaN, it means that driver/race combo doesn't exist in results_clean)
missing_drivers = failed_to_qualify_and_missing_from_results[
    failed_to_qualify_and_missing_from_results['grid'].isna()
]

print(f"Number of drivers who qualified but are missing entirely from results: {len(missing_drivers)}\n")

# 3. Check if your results table uses the 'F' text flag for failed qualifications
if 'positionText' in results_clean.columns:
    failed_to_qualify_in_results = results_clean[results_clean['positionText'] == 'F']
    print(f"Number of drivers explicitly marked as 'F' (Failed to Qualify) in results: {len(failed_to_qualify_in_results)}")
    
    if len(failed_to_qualify_in_results) > 0:
        display(failed_to_qualify_in_results[['raceId', 'driverId', 'grid', 'positionText']].head())
else:
    print("The 'positionText' column was not found in results_clean.")

In [ ]:
import pandas as pd

# 1. Look for drivers explicitly marked with "F" (Failed to qualify) in the results text column
if 'positionText' in results.columns:
    dnq_results = results_clean[results['positionText'] == 'F']
    
    print(f"Found {len(dnq_results)} drivers marked as 'Failed to Qualify' (F) in the results table.")
    
    # 2. Let's look at the actual data rows for these drivers
    if len(dnq_results) > 0:
        # Keep relevant identifiers to check out who they are
        cols_to_show = [col for col in ['raceId', 'driverId', 'grid', 'positionText', 'positionOrder'] if col in results_clean.columns]
        display(dnq_results[cols_to_show].head(10))
else:
    print("The 'positionText' column was not found in your results_clean table.")


In [ ]:
# 1. Count how many drivers are missing Q1, missing Q2, or missing at least one of them
missing_either = f_qualifying_times['q1_s'].isna() | f_qualifying_times['q2_s'].isna()
count_missing_either = missing_either.sum()

count_missing_q1 = f_qualifying_times['q1_s'].isna().sum()
count_missing_q2 = f_qualifying_times['q2_s'].isna().sum()

# 2. Print the final counts
print(f"Total 'F' drivers evaluated: {len(f_qualifying_times)}")
print(f"Count of 'F' drivers missing Q1: {count_missing_q1}")
print(f"Count of 'F' drivers missing Q2: {count_missing_q2}")
print(f"Count of 'F' drivers missing EITHER Q1 or Q2: {count_missing_either}")


In [ ]:
# Count exactly how many rows have 'F' in the positionText column
f_count = (results['positionText'] == 'F').sum()

print(f"Total number of 'F' (Failed to Qualify) entries in results table: {f_count}")
